In [1]:
# ============================================================
# PHASE 3 - MODEL DEVELOPMENT AND MODEL COMPARISON
# ============================================================
#
# Models:
#   1. XGBoost
#   2. Random Forest
#   3. Support Vector Machine
#   4. Logistic Regression
#
# XGBoost:
#   - WRI
#   - Exposure
#   - Vulnerability
#   - Susceptibility
#
# Model comparison:
#   - Susceptibility
#
# Validation:
#   - Stratified 5-fold Cross Validation
#
# Final evaluation:
#   - Independent 20% test set
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import time
import warnings
import joblib
import numpy as np
import pandas as pd

from xgboost import XGBClassifier

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")


# ============================================================
# 2. PROJECT PATHS
# ============================================================

BASE_DIR = os.path.abspath("..")

PROCESSED_DIR = os.path.join(
    BASE_DIR,
    "data",
    "processed"
)

MODEL_DIR = os.path.join(
    BASE_DIR,
    "models"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ============================================================
# 3. LOAD PROCESSED DATA
# ============================================================

print("=" * 70)
print("PHASE 3 - MODEL DEVELOPMENT")
print("=" * 70)

print("\nLoading processed datasets...")


X_train = pd.read_csv(
    os.path.join(
        PROCESSED_DIR,
        "X_train.csv"
    )
)

X_test = pd.read_csv(
    os.path.join(
        PROCESSED_DIR,
        "X_test.csv"
    )
)

y_train = pd.read_csv(
    os.path.join(
        PROCESSED_DIR,
        "y_train.csv"
    )
)

y_test = pd.read_csv(
    os.path.join(
        PROCESSED_DIR,
        "y_test.csv"
    )
)


print("\nData loaded successfully.")

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)


# ============================================================
# 4. TARGET VARIABLES
# ============================================================

TARGETS = [
    "WRI_Category",
    "Exposure_Category",
    "Vulnerability_Category",
    "Susceptibility_Category"
]


print("\nTargets:")

for target in TARGETS:
    print(" -", target)


# ============================================================
# 5. CHECK TARGET COLUMNS
# ============================================================

for target in TARGETS:

    if target not in y_train.columns:

        raise ValueError(
            f"{target} is missing from y_train.csv"
        )

    if target not in y_test.columns:

        raise ValueError(
            f"{target} is missing from y_test.csv"
        )


# ============================================================
# 6. CONVERT TARGETS TO INTEGER
# ============================================================

for target in TARGETS:

    y_train[target] = (
        pd.to_numeric(
            y_train[target],
            errors="coerce"
        )
        .astype(int)
    )

    y_test[target] = (
        pd.to_numeric(
            y_test[target],
            errors="coerce"
        )
        .astype(int)
    )


# ============================================================
# 7. CHECK FEATURES
# ============================================================

# Make sure the training and test feature columns
# are identical and in the same order.

if list(X_train.columns) != list(X_test.columns):

    raise ValueError(
        "Training and testing feature columns do not match."
    )


FEATURE_COLUMNS = list(
    X_train.columns
)


print("\nNumber of features:", len(FEATURE_COLUMNS))


# ============================================================
# 8. XGBOOST PARAMETERS
# ============================================================

# These are reproducible baseline parameters.
#
# IMPORTANT:
# The paper does not provide enough information in the
# available text to claim that these are its exact
# hyperparameters.

XGB_PARAMS = {

    "n_estimators": 200,

    "max_depth": 6,

    "learning_rate": 0.05,

    "subsample": 0.8,

    "colsample_bytree": 0.8,

    "objective": "multi:softprob",

    "num_class": 5,

    "eval_metric": "mlogloss",

    "random_state": 42,

    "n_jobs": -1
}


# ============================================================
# 9. CREATE XGBOOST MODEL FUNCTION
# ============================================================

def create_xgboost():

    return XGBClassifier(
        **XGB_PARAMS
    )


# ============================================================
# 10. STRATIFIED 5-FOLD CV FUNCTION
# ============================================================

def run_cross_validation(
    model,
    X,
    y,
    model_name
):

    """
    Performs stratified 5-fold cross-validation.

    Returns:
        accuracy
        precision
        recall
        F1
        AUC
        training time
    """

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    accuracy_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []
    auc_scores = []
    training_times = []


    for fold, (
        train_index,
        validation_index
    ) in enumerate(
        cv.split(X, y),
        start=1
    ):

        X_fold_train = X.iloc[
            train_index
        ]

        X_fold_validation = X.iloc[
            validation_index
        ]

        y_fold_train = y.iloc[
            train_index
        ]

        y_fold_validation = y.iloc[
            validation_index
        ]


        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        start_time = time.time()

        model.fit(
            X_fold_train,
            y_fold_train
        )

        training_time = (
            time.time()
            - start_time
        )


        # ----------------------------------------------------
        # PREDICT
        # ----------------------------------------------------

        predictions = model.predict(
            X_fold_validation
        )

        probabilities = model.predict_proba(
            X_fold_validation
        )


        # ----------------------------------------------------
        # METRICS
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_fold_validation,
            predictions
        )

        precision = precision_score(
            y_fold_validation,
            predictions,
            average="weighted",
            zero_division=0
        )

        recall = recall_score(
            y_fold_validation,
            predictions,
            average="weighted",
            zero_division=0
        )

        f1 = f1_score(
            y_fold_validation,
            predictions,
            average="weighted",
            zero_division=0
        )


        # Multiclass ROC-AUC
        try:

            auc = roc_auc_score(
                y_fold_validation,
                probabilities,
                multi_class="ovr",
                average="weighted"
            )

        except ValueError:

            auc = np.nan


        accuracy_scores.append(
            accuracy
        )

        precision_scores.append(
            precision
        )

        recall_scores.append(
            recall
        )

        f1_scores.append(
            f1
        )

        auc_scores.append(
            auc
        )

        training_times.append(
            training_time
        )


        print(
            f"    Fold {fold}/5 | "
            f"Accuracy = {accuracy:.4f} | "
            f"F1 = {f1:.4f} | "
            f"AUC = {auc:.4f}"
        )


    return {

        "Model": model_name,

        "CV_Accuracy_Mean":
            np.nanmean(accuracy_scores),

        "CV_Accuracy_STD":
            np.nanstd(accuracy_scores),

        "CV_Precision_Mean":
            np.nanmean(precision_scores),

        "CV_Recall_Mean":
            np.nanmean(recall_scores),

        "CV_F1_Mean":
            np.nanmean(f1_scores),

        "CV_AUC_Mean":
            np.nanmean(auc_scores),

        "CV_Training_Time":
            np.mean(training_times)
    }


# ============================================================
# 11. TRAIN XGBOOST FOR FOUR TARGETS
# ============================================================

print("\n")
print("=" * 70)
print("PART A - XGBOOST FOR FOUR RISK DIMENSIONS")
print("=" * 70)


xgb_cv_results = []

trained_xgb_models = {}


for target in TARGETS:

    print("\n")
    print("-" * 70)
    print("TARGET:", target)
    print("-" * 70)


    y = y_train[target]


    print("\nClass distribution:")

    print(
        y.value_counts(
            sort=False
        ).sort_index()
    )


    # --------------------------------------------------------
    # CROSS VALIDATION
    # --------------------------------------------------------

    model = create_xgboost()


    result = run_cross_validation(
        model,
        X_train,
        y,
        "XGBoost"
    )


    result["Target"] = target


    xgb_cv_results.append(
        result
    )


    # --------------------------------------------------------
    # TRAIN FINAL MODEL
    # --------------------------------------------------------

    print("\nTraining final XGBoost model...")

    final_model = create_xgboost()


    start_time = time.time()

    final_model.fit(
        X_train,
        y
    )

    final_training_time = (
        time.time()
        - start_time
    )


    trained_xgb_models[
        target
    ] = final_model


    # --------------------------------------------------------
    # SAVE MODEL
    # --------------------------------------------------------

    filename = (
        target
        .replace("_Category", "")
        .lower()
        + "_xgboost.pkl"
    )


    model_path = os.path.join(
        MODEL_DIR,
        filename
    )


    joblib.dump(
        final_model,
        model_path
    )


    print(
        "\nModel saved:",
        model_path
    )

    print(
        "Final training time:",
        round(
            final_training_time,
            2
        ),
        "seconds"
    )


# ============================================================
# 12. SUSCEPTIBILITY MODEL COMPARISON
# ============================================================

print("\n")
print("=" * 70)
print("PART B - MODEL COMPARISON")
print("=" * 70)

print(
    "\nComparing models for SUSCEPTIBILITY."
)

print(
    "\nModels:"
)

print(" - XGBoost")
print(" - Random Forest")
print(" - Support Vector Machine")
print(" - Logistic Regression")


# ============================================================
# 13. SUSCEPTIBILITY TARGET
# ============================================================

susceptibility_target = (
    "Susceptibility_Category"
)

y_sus_train = y_train[
    susceptibility_target
]

y_sus_test = y_test[
    susceptibility_target
]


# ============================================================
# 14. DEFINE COMPARISON MODELS
# ============================================================

comparison_models = {

    "XGBoost":
        create_xgboost(),


    "Random Forest":
        RandomForestClassifier(

            n_estimators=200,

            max_depth=None,

            random_state=42,

            n_jobs=-1
        ),


    "SVM":
        Pipeline([

            (
                "scaler",

                StandardScaler()
            ),

            (
                "classifier",

                SVC(

                    kernel="rbf",

                    probability=True,

                    random_state=42
                )
            )
        ]),


    "Logistic Regression":
        Pipeline([

            (
                "scaler",

                StandardScaler()
            ),

            (
                "classifier",

                LogisticRegression(

                    max_iter=2000,

                    random_state=42
                )
            )
        ])
}


# ============================================================
# 15. CROSS-VALIDATION FOR COMPARISON
# ============================================================

comparison_cv_results = []


for model_name, model in comparison_models.items():

    print("\n")
    print("-" * 70)

    print(
        "MODEL:",
        model_name
    )

    print("-" * 70)


    result = run_cross_validation(

        model,

        X_train,

        y_sus_train,

        model_name
    )


    result["Target"] = (
        "Susceptibility"
    )


    comparison_cv_results.append(
        result
    )


# ============================================================
# 16. TRAIN FINAL COMPARISON MODELS
# ============================================================

print("\n")
print("=" * 70)
print("TRAINING FINAL COMPARISON MODELS")
print("=" * 70)


final_comparison_models = {}

holdout_comparison_results = []


for model_name, model in comparison_models.items():

    print(
        "\nTraining:",
        model_name
    )


    start_time = time.time()


    model.fit(
        X_train,
        y_sus_train
    )


    training_time = (
        time.time()
        - start_time
    )


    # --------------------------------------------------------
    # Test predictions
    # --------------------------------------------------------

    predictions = model.predict(
        X_test
    )


    probabilities = model.predict_proba(
        X_test
    )


    # --------------------------------------------------------
    # Test metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_sus_test,
        predictions
    )


    precision = precision_score(
        y_sus_test,
        predictions,
        average="weighted",
        zero_division=0
    )


    recall = recall_score(
        y_sus_test,
        predictions,
        average="weighted",
        zero_division=0
    )


    f1 = f1_score(
        y_sus_test,
        predictions,
        average="weighted",
        zero_division=0
    )


    try:

        auc = roc_auc_score(
            y_sus_test,
            probabilities,
            multi_class="ovr",
            average="weighted"
        )

    except ValueError:

        auc = np.nan


    # --------------------------------------------------------
    # Save results
    # --------------------------------------------------------

    holdout_comparison_results.append({

        "Model":
            model_name,

        "Accuracy":
            accuracy,

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,

        "AUC":
            auc,

        "Training_Time":
            training_time
    })


    # --------------------------------------------------------
    # Save model in memory
    # --------------------------------------------------------

    final_comparison_models[
        model_name
    ] = model


    # --------------------------------------------------------
    # Print results
    # --------------------------------------------------------

    print(
        "Accuracy :",
        round(accuracy, 4)
    )

    print(
        "Precision:",
        round(precision, 4)
    )

    print(
        "Recall   :",
        round(recall, 4)
    )

    print(
        "F1       :",
        round(f1, 4)
    )

    print(
        "AUC      :",
        round(auc, 4)
    )


# ============================================================
# 17. CREATE COMPARISON TABLE
# ============================================================

comparison_results_df = pd.DataFrame(
    holdout_comparison_results
)


print("\n")
print("=" * 70)
print("SUSCEPTIBILITY MODEL COMPARISON")
print("=" * 70)

print(
    comparison_results_df.to_string(
        index=False
    )
)


# ============================================================
# 18. SAVE COMPARISON RESULTS
# ============================================================

comparison_file = os.path.join(
    OUTPUT_DIR,
    "susceptibility_model_comparison.csv"
)


comparison_results_df.to_csv(
    comparison_file,
    index=False
)


# ============================================================
# 19. SAVE CV RESULTS
# ============================================================

xgb_cv_df = pd.DataFrame(
    xgb_cv_results
)


xgb_cv_file = os.path.join(
    OUTPUT_DIR,
    "xgboost_cross_validation_results.csv"
)


xgb_cv_df.to_csv(
    xgb_cv_file,
    index=False
)


comparison_cv_df = pd.DataFrame(
    comparison_cv_results
)


comparison_cv_file = os.path.join(
    OUTPUT_DIR,
    "model_comparison_cross_validation.csv"
)


comparison_cv_df.to_csv(
    comparison_cv_file,
    index=False
)


# ============================================================
# 20. SAVE FINAL MODELS
# ============================================================

all_models = {

    "xgboost_four_targets":
        trained_xgb_models,

    "susceptibility_comparison":
        final_comparison_models
}


all_models_file = os.path.join(
    MODEL_DIR,
    "phase3_all_models.pkl"
)


joblib.dump(
    all_models,
    all_models_file
)


# ============================================================
# 21. SAVE SUSCEPTIBILITY PREDICTIONS
# ============================================================

susceptibility_predictions = pd.DataFrame({

    "Actual":
        y_sus_test.values
})


for model_name, model in final_comparison_models.items():

    susceptibility_predictions[
        model_name + "_Prediction"
    ] = model.predict(
        X_test
    )


prediction_file = os.path.join(
    OUTPUT_DIR,
    "susceptibility_model_predictions.csv"
)


susceptibility_predictions.to_csv(
    prediction_file,
    index=False
)


# ============================================================
# 22. PRINT XGBOOST CV RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("XGBOOST - FOUR TARGET CROSS-VALIDATION RESULTS")
print("=" * 70)

print(
    xgb_cv_df[
        [
            "Target",
            "CV_Accuracy_Mean",
            "CV_Accuracy_STD",
            "CV_Precision_Mean",
            "CV_Recall_Mean",
            "CV_F1_Mean",
            "CV_AUC_Mean"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("PHASE 3 COMPLETED SUCCESSFULLY")
print("=" * 70)


print("\nXGBoost models trained for:")

for target in TARGETS:

    print(
        " -",
        target
    )


print("\nSusceptibility comparison models:")

for model_name in comparison_models:

    print(
        " -",
        model_name
    )


print("\nFiles created:")

print(
    "\n1.",
    comparison_file
)

print(
    "2.",
    xgb_cv_file
)

print(
    "3.",
    comparison_cv_file
)

print(
    "4.",
    prediction_file
)

print(
    "5.",
    all_models_file
)


print("\n")
print("=" * 70)
print("READY FOR PHASE 4 - MODEL EVALUATION")
print("=" * 70)

PHASE 3 - MODEL DEVELOPMENT

Loading processed datasets...

Data loaded successfully.
X_train shape: (4014, 242)
X_test shape : (1004, 242)
y_train shape: (4014, 4)
y_test shape : (1004, 4)

Targets:
 - WRI_Category
 - Exposure_Category
 - Vulnerability_Category
 - Susceptibility_Category

Number of features: 242


PART A - XGBOOST FOR FOUR RISK DIMENSIONS


----------------------------------------------------------------------
TARGET: WRI_Category
----------------------------------------------------------------------

Class distribution:
WRI_Category
0    806
1    801
2    802
3    802
4    803
Name: count, dtype: int64
    Fold 1/5 | Accuracy = 0.9402 | F1 = 0.9403 | AUC = 0.9960
    Fold 2/5 | Accuracy = 0.9601 | F1 = 0.9600 | AUC = 0.9953
    Fold 3/5 | Accuracy = 0.9738 | F1 = 0.9738 | AUC = 0.9984
    Fold 4/5 | Accuracy = 0.9651 | F1 = 0.9652 | AUC = 0.9975
    Fold 5/5 | Accuracy = 0.9514 | F1 = 0.9512 | AUC = 0.9945

Training final XGBoost model...

Model saved: c:\Users\mvars